### **IMPORTO LIBRERIE**

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import statsmodels.formula.api as smf
from statsmodels.graphics.gofplots import qqplot
from statsmodels.nonparametric.smoothers_lowess import lowess
from statsmodels.stats.diagnostic import het_breuschpagan
from statsmodels.stats.outliers_influence import variance_inflation_factor

### **IMPORTO DATI**

In [ ]:
processed_dir = Path("datas/processed")

software_df = pd.read_csv(processed_dir / "dataset_software.csv")
hardware_df = pd.read_csv(processed_dir / "dataset_hardware.csv")

### **REGRESSIONE OLS POOLED ROE**

Stima pooled su imprese software e hardware, includendo la dummy `software` tra i regressori.

In [ ]:
software_df = software_df.assign(software=1)
hardware_df = hardware_df.assign(software=0)

df_pooled = pd.concat([software_df, hardware_df], ignore_index=True)

variabili_modello = [
    "roe_medio",
    "debt_equity_medio",
    "liquidita_media",
    "log_attivo_medio",
    "log_eta_azienda",
    "software",
]

df_ols = df_pooled[variabili_modello].dropna().copy()

formula = (
    "roe_medio ~ debt_equity_medio + liquidita_media + log_attivo_medio "
    "+ log_eta_azienda + software"
)

modello_ols = smf.ols(formula=formula, data=df_ols).fit()

print(f"Osservazioni pooled: {len(df_pooled)}")
print(f"Osservazioni usate nella regressione: {len(df_ols)}")
print(modello_ols.summary())

### **TEST DELLE IPOTESI DEL MODELLO OLS ROE**

Primo controllo diagnostico: il grafico residui vs fitted permette di valutare visivamente linearità e omoschedasticità. In assenza di problemi evidenti, i residui dovrebbero distribuirsi in modo casuale intorno alla linea orizzontale dello zero, senza curvature sistematiche o forma a imbuto.

In [ ]:
residui_roe = modello_ols.resid
valori_stimati_roe = modello_ols.fittedvalues
lowess_roe = lowess(residui_roe, valori_stimati_roe, frac=0.6, return_sorted=True)

grafici_dir = Path("regressione_export/grafici")
grafici_dir.mkdir(parents=True, exist_ok=True)

fig, ax = plt.subplots(figsize=(8, 5))

ax.scatter(
    valori_stimati_roe,
    residui_roe,
    alpha=0.7,
    edgecolor="white",
    linewidth=0.5,
)
ax.axhline(0, color="black", linestyle="--", linewidth=1)
ax.plot(
    lowess_roe[:, 0],
    lowess_roe[:, 1],
    color="#c44e52",
    linewidth=2,
    label="LOWESS",
)

ax.set_title("Residui vs fitted - modello OLS ROE")
ax.set_xlabel("Valori stimati")
ax.set_ylabel("Residui")
ax.legend()
ax.grid(alpha=0.25)

residui_fitted_roe_path = grafici_dir / "roe_residui_vs_fitted.png"
fig.savefig(residui_fitted_roe_path, dpi=300, bbox_inches="tight")

print(f"Grafico esportato in: {residui_fitted_roe_path}")
plt.show()

### **Q-Q PLOT DEI RESIDUI DEL MODELLO ROE**

Il Q-Q plot confronta la distribuzione dei residui con quella normale teorica. Se l'ipotesi di normalità è plausibile, i punti dovrebbero disporsi approssimativamente lungo la linea diagonale.

In [ ]:
fig, ax = plt.subplots(figsize=(6, 6))

qqplot(residui_roe, line="s", ax=ax)

ax.set_title("Q-Q plot dei residui - modello OLS ROE")
ax.set_xlabel("Quantili teorici")
ax.set_ylabel("Quantili campionari")
ax.grid(alpha=0.25)

qq_plot_roe_path = grafici_dir / "roe_qq_plot_residui.png"
fig.savefig(qq_plot_roe_path, dpi=300, bbox_inches="tight")

print(f"Grafico esportato in: {qq_plot_roe_path}")
plt.show()

### **VIF DEI REGRESSORI DEL MODELLO ROE**

Il Variance Inflation Factor misura la multicollinearità tra i regressori. Valori elevati indicano che una variabile esplicativa è fortemente spiegata dalle altre variabili del modello, aumentando l'incertezza delle stime dei coefficienti.

In [ ]:
matrice_regressori_roe = modello_ols.model.exog
nomi_regressori_roe = modello_ols.model.exog_names

vif_roe = pd.DataFrame(
    {
        "Variabile": nomi_regressori_roe,
        "VIF": [
            variance_inflation_factor(matrice_regressori_roe, i)
            for i in range(matrice_regressori_roe.shape[1])
        ],
    }
)

vif_roe = vif_roe[vif_roe["Variabile"] != "Intercept"].reset_index(drop=True)

tex_dir = Path("regressione_export/tex")
tex_dir.mkdir(parents=True, exist_ok=True)

vif_roe_path = tex_dir / "vif_roe.tex"
vif_roe.to_latex(
    vif_roe_path,
    index=False,
    float_format="%.4f",
    caption="Variance Inflation Factor dei regressori del modello OLS pooled sul ROE",
    label="tab:vif_roe",
)

print("Variance Inflation Factor - modello OLS ROE")
print(f"Tabella esportata in: {vif_roe_path}")
display(vif_roe)

### **TEST DI BREUSCH-PAGAN SUL MODELLO ROE**

Test di eteroschedasticità sui residui della regressione OLS pooled con `roe_medio` come variabile dipendente.

In [ ]:
bp_test = het_breuschpagan(modello_ols.resid, modello_ols.model.exog)

bp_results = pd.Series(
    bp_test,
    index=[
        "LM statistic",
        "LM p-value",
        "F statistic",
        "F p-value",
    ],
    name="Valore",
)
bp_results_latex = bp_results.to_frame()

bp_latex_path = Path("regressione_export/tex") / "breusch_pagan_roe.tex"
bp_latex_path.parent.mkdir(parents=True, exist_ok=True)
bp_results_latex.to_latex(
    bp_latex_path,
    float_format="%.4f",
    caption="Risultati del test di Breusch-Pagan sui residui OLS del modello ROE",
    label="tab:breusch_pagan_roe",
)

print("Test di Breusch-Pagan sui residui OLS")
print(f"Tabella esportata in: {bp_latex_path}")
print(bp_results)


### **DISTANZA DI COOK DEL MODELLO ROE**

La Distanza di Cook consente di individuare osservazioni potenzialmente influenti, cioè osservazioni che possono incidere in modo rilevante sulla stima dei coefficienti. Come soglia operativa viene usato il criterio `4/n`, dove `n` è il numero di osservazioni del modello.

In [ ]:
influenza_roe = modello_ols.get_influence()
distanza_cook_roe = influenza_roe.cooks_distance[0]
soglia_cook_roe = 4 / len(df_ols)

cook_roe = pd.DataFrame(
    {
        "Indice osservazione": df_ols.index,
        "Distanza di Cook": distanza_cook_roe,
        "Soglia 4/n": soglia_cook_roe,
        "Potenzialmente influente": distanza_cook_roe > soglia_cook_roe,
    }
)

cook_roe_top = cook_roe.sort_values("Distanza di Cook", ascending=False).head(10)

fig, ax = plt.subplots(figsize=(9, 5))

ax.scatter(
    range(len(distanza_cook_roe)),
    distanza_cook_roe,
    alpha=0.75,
    s=28,
)
ax.axhline(
    soglia_cook_roe,
    color="#c44e52",
    linestyle="--",
    linewidth=1.5,
    label=f"Soglia 4/n = {soglia_cook_roe:.4f}",
)

ax.set_title("Distanza di Cook - modello OLS ROE")
ax.set_xlabel("Osservazione")
ax.set_ylabel("Distanza di Cook")
ax.legend()
ax.grid(alpha=0.25)

cook_roe_path = grafici_dir / "roe_distanza_cook.png"
fig.savefig(cook_roe_path, dpi=300, bbox_inches="tight")

print(f"Soglia 4/n: {soglia_cook_roe:.4f}")
print(
    "Osservazioni potenzialmente influenti: "
    f"{cook_roe['Potenzialmente influente'].sum()}"
)
print(f"Grafico esportato in: {cook_roe_path}")
display(cook_roe_top)
plt.show()

### **REGRESSIONE OLS ROE SENZA OSSERVAZIONI INFLUENTI**

Il modello viene ristimato eliminando le osservazioni individuate come potenzialmente influenti dalla Distanza di Cook. I coefficienti ottenuti vengono poi confrontati con quelli del modello OLS ROE completo.

In [ ]:
osservazioni_influenti_roe = cook_roe.loc[
    cook_roe["Potenzialmente influente"],
    "Indice osservazione",
]

df_ols_senza_influenti = df_ols.drop(
    index=osservazioni_influenti_roe
).copy()

modello_ols_senza_influenti = smf.ols(
    formula=formula,
    data=df_ols_senza_influenti,
).fit()

confronto_coefficienti_roe = pd.DataFrame(
    {
        "Coefficiente modello completo": modello_ols.params,
        "Coefficiente senza influenti": modello_ols_senza_influenti.params,
    }
)
confronto_coefficienti_roe["Differenza"] = (
    confronto_coefficienti_roe["Coefficiente senza influenti"]
    - confronto_coefficienti_roe["Coefficiente modello completo"]
)
confronto_coefficienti_roe["Differenza assoluta"] = (
    confronto_coefficienti_roe["Differenza"].abs()
)

confronto_coefficienti_roe_latex_path = (
    Path("regressione_export/tex")
    / "confronto_coefficienti_roe_senza_influenti.tex"
)
confronto_coefficienti_roe_latex_path.parent.mkdir(parents=True, exist_ok=True)
confronto_coefficienti_roe.to_latex(
    confronto_coefficienti_roe_latex_path,
    float_format="%.6f",
    caption="Confronto dei coefficienti ROE prima e dopo l'esclusione delle osservazioni influenti secondo Cook",
    label="tab:confronto_coefficienti_roe_senza_influenti",
)

print(f"Osservazioni rimosse: {len(osservazioni_influenti_roe)}")
print(
    "Osservazioni usate dopo l'esclusione: "
    f"{len(df_ols_senza_influenti)}"
)

display(confronto_coefficienti_roe)
print(modello_ols_senza_influenti.summary())

### **REGRESSIONE OLS POOLED ROE CON ERRORI STANDARD ROBUSTI HC3**

A seguito del test di Breusch-Pagan, il modello ROE viene ristimato con errori standard robusti HC3.

In [ ]:
modello_roe_pooled = smf.ols(formula=formula, data=df_ols).fit(cov_type="HC3")

print(modello_roe_pooled.summary())

### **ESPORTAZIONE TABELLA COEFFICIENTI ROE IN LATEX**

In [ ]:
export_dir = Path("regressione_export/tex")
export_dir.mkdir(parents=True, exist_ok=True)

conf_int = modello_roe_pooled.conf_int()
tabella_coefficienti = pd.DataFrame(
    {
        "Coefficiente": modello_roe_pooled.params,
        "Errore standard HC3": modello_roe_pooled.bse,
        "t": modello_roe_pooled.tvalues,
        "p-value": modello_roe_pooled.pvalues,
        "IC 95% inf.": conf_int[0],
        "IC 95% sup.": conf_int[1],
    }
)

latex_path = export_dir / "modello_roe_pooled.tex"
tabella_coefficienti.to_latex(
    latex_path,
    float_format="%.4f",
    caption="Coefficienti del modello OLS pooled sul ROE con errori standard robusti HC3",
    label="tab:modello_roe_pooled",
)

print(f"Tabella esportata in: {latex_path}")
display(tabella_coefficienti)


### **REGRESSIONE OLS POOLED ROA**

Stima pooled su imprese software e hardware, con `roa_medio` come variabile dipendente.

In [ ]:
variabili_modello_roa = [
    "roa_medio",
    "debt_equity_medio",
    "liquidita_media",
    "log_attivo_medio",
    "log_eta_azienda",
    "software",
]

df_ols_roa = df_pooled[variabili_modello_roa].dropna().copy()

formula_roa = (
    "roa_medio ~ debt_equity_medio + liquidita_media + log_attivo_medio "
    "+ log_eta_azienda + software"
)

modello_ols_roa = smf.ols(formula=formula_roa, data=df_ols_roa).fit()

print(f"Osservazioni pooled: {len(df_pooled)}")
print(f"Osservazioni usate nella regressione ROA: {len(df_ols_roa)}")
print(modello_ols_roa.summary())

### **TEST DELLE IPOTESI DEL MODELLO OLS ROA**

Primo controllo diagnostico: il grafico residui vs fitted permette di valutare visivamente linearità e omoschedasticità. In assenza di problemi evidenti, i residui dovrebbero distribuirsi in modo casuale intorno alla linea orizzontale dello zero, senza curvature sistematiche o forma a imbuto.

In [ ]:
residui_roa = modello_ols_roa.resid
valori_stimati_roa = modello_ols_roa.fittedvalues
lowess_roa = lowess(residui_roa, valori_stimati_roa, frac=0.6, return_sorted=True)

grafici_dir = Path("regressione_export/grafici")
grafici_dir.mkdir(parents=True, exist_ok=True)

fig, ax = plt.subplots(figsize=(8, 5))

ax.scatter(
    valori_stimati_roa,
    residui_roa,
    alpha=0.7,
    edgecolor="white",
    linewidth=0.5,
)
ax.axhline(0, color="black", linestyle="--", linewidth=1)
ax.plot(
    lowess_roa[:, 0],
    lowess_roa[:, 1],
    color="#c44e52",
    linewidth=2,
    label="LOWESS",
)

ax.set_title("Residui vs fitted - modello OLS ROA")
ax.set_xlabel("Valori stimati")
ax.set_ylabel("Residui")
ax.legend()
ax.grid(alpha=0.25)

residui_fitted_roa_path = grafici_dir / "roa_residui_vs_fitted.png"
fig.savefig(residui_fitted_roa_path, dpi=300, bbox_inches="tight")

print(f"Grafico esportato in: {residui_fitted_roa_path}")
plt.show()

### **Q-Q PLOT DEI RESIDUI DEL MODELLO ROA**

Il Q-Q plot confronta la distribuzione dei residui con quella normale teorica. Se l'ipotesi di normalità è plausibile, i punti dovrebbero disporsi approssimativamente lungo la linea diagonale.

In [ ]:
fig, ax = plt.subplots(figsize=(6, 6))

qqplot(residui_roa, line="s", ax=ax)

ax.set_title("Q-Q plot dei residui - modello OLS ROA")
ax.set_xlabel("Quantili teorici")
ax.set_ylabel("Quantili campionari")
ax.grid(alpha=0.25)

qq_plot_roa_path = grafici_dir / "roa_qq_plot_residui.png"
fig.savefig(qq_plot_roa_path, dpi=300, bbox_inches="tight")

print(f"Grafico esportato in: {qq_plot_roa_path}")
plt.show()

### **VIF DEI REGRESSORI DEL MODELLO ROA**

Il Variance Inflation Factor misura la multicollinearità tra i regressori. Valori elevati indicano che una variabile esplicativa è fortemente spiegata dalle altre variabili del modello, aumentando l'incertezza delle stime dei coefficienti.

In [ ]:
matrice_regressori_roa = modello_ols_roa.model.exog
nomi_regressori_roa = modello_ols_roa.model.exog_names

vif_roa = pd.DataFrame(
    {
        "Variabile": nomi_regressori_roa,
        "VIF": [
            variance_inflation_factor(matrice_regressori_roa, i)
            for i in range(matrice_regressori_roa.shape[1])
        ],
    }
)

vif_roa = vif_roa[vif_roa["Variabile"] != "Intercept"].reset_index(drop=True)

tex_dir = Path("regressione_export/tex")
tex_dir.mkdir(parents=True, exist_ok=True)

vif_roa_path = tex_dir / "vif_roa.tex"
vif_roa.to_latex(
    vif_roa_path,
    index=False,
    float_format="%.4f",
    caption="Variance Inflation Factor dei regressori del modello OLS pooled sul ROA",
    label="tab:vif_roa",
)

print("Variance Inflation Factor - modello OLS ROA")
print(f"Tabella esportata in: {vif_roa_path}")
display(vif_roa)

### **TEST DI BREUSCH-PAGAN SUL MODELLO ROA**

Test di eteroschedasticità sui residui della regressione OLS pooled con `roa_medio` come variabile dipendente.

In [ ]:
bp_test_roa = het_breuschpagan(modello_ols_roa.resid, modello_ols_roa.model.exog)

bp_results_roa = pd.Series(
    bp_test_roa,
    index=[
        "LM statistic",
        "LM p-value",
        "F statistic",
        "F p-value",
    ],
    name="Valore",
)
bp_results_roa_latex = bp_results_roa.to_frame()

bp_latex_path_roa = Path("regressione_export/tex") / "breusch_pagan_roa.tex"
bp_latex_path_roa.parent.mkdir(parents=True, exist_ok=True)
bp_results_roa_latex.to_latex(
    bp_latex_path_roa,
    float_format="%.4f",
    caption="Risultati del test di Breusch-Pagan sui residui OLS del modello ROA",
    label="tab:breusch_pagan_roa",
)

print("Test di Breusch-Pagan sui residui OLS - ROA")
print(f"Tabella esportata in: {bp_latex_path_roa}")
print(bp_results_roa)


### **DISTANZA DI COOK DEL MODELLO ROA**

La Distanza di Cook consente di individuare osservazioni potenzialmente influenti, cioè osservazioni che possono incidere in modo rilevante sulla stima dei coefficienti. Come soglia operativa viene usato il criterio `4/n`, dove `n` è il numero di osservazioni del modello.

In [ ]:
influenza_roa = modello_ols_roa.get_influence()
distanza_cook_roa = influenza_roa.cooks_distance[0]
soglia_cook_roa = 4 / len(df_ols_roa)

cook_roa = pd.DataFrame(
    {
        "Indice osservazione": df_ols_roa.index,
        "Distanza di Cook": distanza_cook_roa,
        "Soglia 4/n": soglia_cook_roa,
        "Potenzialmente influente": distanza_cook_roa > soglia_cook_roa,
    }
)

cook_roa_top = cook_roa.sort_values("Distanza di Cook", ascending=False).head(10)

fig, ax = plt.subplots(figsize=(9, 5))

ax.scatter(
    range(len(distanza_cook_roa)),
    distanza_cook_roa,
    alpha=0.75,
    s=28,
)
ax.axhline(
    soglia_cook_roa,
    color="#c44e52",
    linestyle="--",
    linewidth=1.5,
    label=f"Soglia 4/n = {soglia_cook_roa:.4f}",
)

ax.set_title("Distanza di Cook - modello OLS ROA")
ax.set_xlabel("Osservazione")
ax.set_ylabel("Distanza di Cook")
ax.legend()
ax.grid(alpha=0.25)

cook_roa_path = grafici_dir / "roa_distanza_cook.png"
fig.savefig(cook_roa_path, dpi=300, bbox_inches="tight")

print(f"Soglia 4/n: {soglia_cook_roa:.4f}")
print(
    "Osservazioni potenzialmente influenti: "
    f"{cook_roa['Potenzialmente influente'].sum()}"
)
print(f"Grafico esportato in: {cook_roa_path}")
display(cook_roa_top)
plt.show()

### **REGRESSIONE OLS ROA SENZA OSSERVAZIONI INFLUENTI**

Il modello viene ristimato eliminando le osservazioni individuate come potenzialmente influenti dalla Distanza di Cook. I coefficienti ottenuti vengono poi confrontati con quelli del modello OLS ROA completo.

In [ ]:
osservazioni_influenti_roa = cook_roa.loc[
    cook_roa["Potenzialmente influente"],
    "Indice osservazione",
]

df_ols_roa_senza_influenti = df_ols_roa.drop(
    index=osservazioni_influenti_roa
).copy()

modello_ols_roa_senza_influenti = smf.ols(
    formula=formula_roa,
    data=df_ols_roa_senza_influenti,
).fit()

confronto_coefficienti_roa = pd.DataFrame(
    {
        "Coefficiente modello completo": modello_ols_roa.params,
        "Coefficiente senza influenti": modello_ols_roa_senza_influenti.params,
    }
)
confronto_coefficienti_roa["Differenza"] = (
    confronto_coefficienti_roa["Coefficiente senza influenti"]
    - confronto_coefficienti_roa["Coefficiente modello completo"]
)
confronto_coefficienti_roa["Differenza assoluta"] = (
    confronto_coefficienti_roa["Differenza"].abs()
)

confronto_coefficienti_roa_latex_path = (
    Path("regressione_export/tex")
    / "confronto_coefficienti_roa_senza_influenti.tex"
)
confronto_coefficienti_roa_latex_path.parent.mkdir(parents=True, exist_ok=True)
confronto_coefficienti_roa.to_latex(
    confronto_coefficienti_roa_latex_path,
    float_format="%.6f",
    caption="Confronto dei coefficienti ROA prima e dopo l'esclusione delle osservazioni influenti secondo Cook",
    label="tab:confronto_coefficienti_roa_senza_influenti",
)

print(f"Osservazioni rimosse: {len(osservazioni_influenti_roa)}")
print(
    "Osservazioni usate dopo l'esclusione: "
    f"{len(df_ols_roa_senza_influenti)}"
)

display(confronto_coefficienti_roa)
print(modello_ols_roa_senza_influenti.summary())

### **REGRESSIONE OLS POOLED ROA CON ERRORI STANDARD ROBUSTI HC3**

A seguito del test di Breusch-Pagan, il modello ROA viene ristimato con errori standard robusti HC3.

In [ ]:
modello_roa_pooled = smf.ols(formula=formula_roa, data=df_ols_roa).fit(cov_type="HC3")

print(modello_roa_pooled.summary())

### **ESPORTAZIONE TABELLA COEFFICIENTI ROA IN LATEX**

In [ ]:
export_dir = Path("regressione_export/tex")
export_dir.mkdir(parents=True, exist_ok=True)

conf_int_roa = modello_roa_pooled.conf_int()
tabella_coefficienti_roa = pd.DataFrame(
    {
        "Coefficiente": modello_roa_pooled.params,
        "Errore standard HC3": modello_roa_pooled.bse,
        "t": modello_roa_pooled.tvalues,
        "p-value": modello_roa_pooled.pvalues,
        "IC 95% inf.": conf_int_roa[0],
        "IC 95% sup.": conf_int_roa[1],
    }
)

latex_path_roa = export_dir / "modello_roa_pooled.tex"
tabella_coefficienti_roa.to_latex(
    latex_path_roa,
    float_format="%.4f",
    caption="Coefficienti del modello OLS pooled sul ROA con errori standard robusti HC3",
    label="tab:modello_roa_pooled",
)

print(f"Tabella esportata in: {latex_path_roa}")
display(tabella_coefficienti_roa)
